# Preparação da base analítica

Como reunir pedidos, clientes, itens e avaliações sem duplicar pedidos? Prepara e valida as bases detalhadas para as análises.

In [1]:
# Importa o pandas e permite visualizar todas as colunas dos DataFrames.
import pandas as pd
pd.set_option('display.max_columns', None)


In [2]:
from pathlib import Path

# Procura a raiz a partir da pasta atual; funciona também nas cópias em outputs/notebooks.
RAIZ_PROJETO = next(
    (pasta for pasta in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (pasta / 'executar_pipeline.py').is_file() and (pasta / 'src').is_dir()),
    None
)
if RAIZ_PROJETO is None:
    raise FileNotFoundError('Execute o notebook dentro da pasta do projeto ou de uma subpasta.')

# Carrega as tabelas originais sem modificar os arquivos da pasta raw.
CAMINHO_DADOS = RAIZ_PROJETO / 'data' / 'raw'

df_order_reviews_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_order_reviews_dataset.csv')
df_customers_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_customers_dataset.csv')
df_order_items_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_order_items_dataset.csv')
df_orders_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_orders_dataset.csv')
df_products_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_products_dataset.csv')
df_sellers_raw = pd.read_csv(f'{CAMINHO_DADOS}/olist_sellers_dataset.csv')


In [3]:
# Cria uma cópia das reviews para preservar o DataFrame original.
df_reviews_por_pedido = (
    df_order_reviews_raw.copy()
)

# Converte as datas; valores inválidos são registrados como NaT.
for coluna in ['review_creation_date', 'review_answer_timestamp']:
    df_reviews_por_pedido[coluna] = pd.to_datetime(
        df_reviews_por_pedido[coluna],
        errors='coerce'
    ).astype('datetime64[ns]')


df_reviews_por_pedido['review_score'] = (
    pd.to_numeric(
    df_reviews_por_pedido['review_score'], errors='coerce')
)

# Usa a data da resposta e, quando ausente, a data de criação da review.
df_reviews_por_pedido['review_event_timestamp'] = (
    df_reviews_por_pedido['review_answer_timestamp'].combine_first(
        df_reviews_por_pedido['review_creation_date'])
)

# Mantém notas válidas e uma única ocorrência por pedido e por review.
df_reviews_por_pedido = (
    df_reviews_por_pedido
    .dropna(subset=['order_id', 'review_score'])
    .loc[lambda df: df['review_score'].between(1, 5)]
    .sort_values(['order_id', 'review_event_timestamp'])
    .drop_duplicates(subset=['order_id'], keep='first')
    .drop_duplicates(subset=['review_id'], keep='first')
    .reset_index(drop=True)
    .copy()
)

# Indicadores booleanos permitem contar cada classe com sum().
df_reviews_por_pedido['review_ruim'] = df_reviews_por_pedido['review_score'].le(2)
df_reviews_por_pedido['review_neutra'] = df_reviews_por_pedido['review_score'].eq(3)
df_reviews_por_pedido['review_boa'] = df_reviews_por_pedido['review_score'].ge(4)

# Confirma a granularidade e a classificação das avaliações.
assert df_reviews_por_pedido['order_id'].is_unique
assert df_reviews_por_pedido['order_id'].notna().all()
assert df_reviews_por_pedido['review_score'].notna().all()
assert df_reviews_por_pedido['review_score'].between(1, 5).all()
assert (
    df_reviews_por_pedido[['review_ruim', 'review_neutra', 'review_boa']]
    .sum(axis=1)
    .eq(1)
    .all()
)


In [4]:
# Valida as chaves antes do merge para não ocultar duplicidades.
assert df_orders_raw['order_id'].is_unique
assert df_orders_raw['order_id'].notna().all()
assert df_orders_raw['customer_id'].notna().all()
assert df_customers_raw['customer_id'].is_unique
assert df_customers_raw['customer_id'].notna().all()

# Enriquece cada pedido com a identificação e a localização do cliente.
df_pedidos_clientes = (
    df_orders_raw
    .merge(
        df_customers_raw,
        on='customer_id',
        how='left',
        validate='1:1'
    )
    .copy()
)

# Padroniza como datetime todas as datas do ciclo do pedido.
for coluna in ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 
               'order_delivered_customer_date', 'order_estimated_delivery_date']:
    df_pedidos_clientes[coluna] = pd.to_datetime(
        df_pedidos_clientes[coluna],
        errors='coerce'
    ).astype('datetime64[ns]')

# Verifica que o merge preservou uma linha para cada pedido original.
assert len(df_pedidos_clientes) == len(df_orders_raw)
assert df_pedidos_clientes['order_id'].is_unique
assert df_pedidos_clientes['order_id'].notna().all()
assert df_pedidos_clientes['customer_id'].is_unique


In [5]:
# Verifica as chaves das tabelas usadas para enriquecer os itens.
assert df_sellers_raw['seller_id'].is_unique
assert df_products_raw['product_id'].is_unique
assert df_sellers_raw['seller_id'].notna().all()
assert df_products_raw['product_id'].notna().all()
assert df_order_items_raw['order_id'].notna().all()
assert df_order_items_raw['order_item_id'].notna().all()
assert df_order_items_raw['product_id'].notna().all()
assert df_order_items_raw['seller_id'].notna().all()


# Adiciona atributos de produto e vendedor sem alterar o nível do item.
df_itens_pedidos = (
    df_order_items_raw
    .merge(
        df_products_raw,
        on='product_id',
        how='left',
        validate='m:1'
    )
    .merge(
        df_sellers_raw,
        on='seller_id',
        how='left',
        validate='m:1'
    )
    .copy()
)


df_itens_pedidos['shipping_limit_date'] = pd.to_datetime(
    df_itens_pedidos['shipping_limit_date'],
    errors='coerce'
).astype('datetime64[ns]')


# Confirma a preservação das linhas e da chave order_id + order_item_id.
assert len(df_itens_pedidos) == len(df_order_items_raw)
assert df_itens_pedidos['order_id'].notna().all()
assert df_itens_pedidos['price'].notna().all()
assert df_itens_pedidos['freight_value'].notna().all()
assert df_itens_pedidos['price'].ge(0).all()
assert df_itens_pedidos['freight_value'].ge(0).all()
assert (
    df_itens_pedidos[['order_id', 'order_item_id']]
    .duplicated()
    .sum() == 0
)


In [6]:
# Calcula o volume cadastrado do produto em cada linha de item.
df_itens_pedidos['volume_item_cm3'] = (
    df_itens_pedidos['product_length_cm']
    * df_itens_pedidos['product_height_cm']
    * df_itens_pedidos['product_width_cm']
)

# Os indicadores preservam a informação de que existem atributos ausentes.
df_itens_pedidos['dimensoes_ausentes'] = (
    df_itens_pedidos[
        [
            'product_length_cm',
            'product_height_cm',
            'product_width_cm'
        ]
    ]
    .isna()
    .any(axis=1)
)

df_itens_pedidos['peso_ausente'] = (
    df_itens_pedidos['product_weight_g'].isna()
)

df_itens_pedidos['categoria_ausente'] = (
    df_itens_pedidos['product_category_name'].isna()
)


In [7]:
# Consolida os diversos itens para obter exatamente uma linha por pedido.
df_resumo_itens_por_pedido = (
    df_itens_pedidos
    .groupby('order_id')
    .agg(
        quantidade_itens=(
            'order_item_id',
            'nunique'
        ),
        quantidade_produtos=(
            'product_id',
            'nunique'
        ),
        quantidade_vendedores=(
            'seller_id',
            'nunique'
        ),
        quantidade_categorias=(
            'product_category_name',
            'nunique'
        ),
        valor_produtos=(
            'price',
            'sum'
        ),
        valor_frete=(
            'freight_value',
            'sum'
        ),
        # min_count=1 evita transformar grupos totalmente ausentes em zero.
        peso_total_g=(
            'product_weight_g',
            lambda valores: valores.sum(min_count=1)
        ),
        volume_total_cm3=(
            'volume_item_cm3',
            lambda valores: valores.sum(min_count=1)
        ),
        possui_dimensoes_ausentes=(
            'dimensoes_ausentes',
            'any'
        ),
        possui_peso_ausente=(
            'peso_ausente',
            'any'
        ),
        possui_categoria_ausente=(
            'categoria_ausente',
            'any'
        )
    )
    .reset_index()
    .copy()
)

df_resumo_itens_por_pedido.head()


,order_id,quantidade_itens,quantidade_produtos,quantidade_vendedores,quantidade_categorias,valor_produtos,valor_frete,peso_total_g,volume_total_cm3,possui_dimensoes_ausentes,possui_peso_ausente,possui_categoria_ausente
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,1,58.90,13.29,650.0,3528.0,False,False,False
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,1,239.90,19.93,30000.0,60000.0,False,False,False
2,000229ec398224ef6ca0657da4fc703e,1,1,1,1,199.00,17.87,3050.0,14157.0,False,False,False
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,1,12.99,12.79,200.0,2400.0,False,False,False
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,1,199.90,18.14,3750.0,42000.0,False,False,False


In [8]:
# Cria métricas financeiras e indicadores de composição do pedido.
df_resumo_itens_por_pedido['valor_total'] = (
    df_resumo_itens_por_pedido['valor_produtos']
    + df_resumo_itens_por_pedido['valor_frete']
)

df_resumo_itens_por_pedido['pct_frete'] = (
    df_resumo_itens_por_pedido['valor_frete']
    / df_resumo_itens_por_pedido[
        'valor_produtos'
    ].replace(0, pd.NA)
)

df_resumo_itens_por_pedido['pedido_um_produto'] = (
    df_resumo_itens_por_pedido[
        'quantidade_produtos'
    ].eq(1)
)

df_resumo_itens_por_pedido['pedido_um_vendedor'] = (
    df_resumo_itens_por_pedido[
        'quantidade_vendedores'
    ].eq(1)
)

# Valida a granularidade e as quantidades geradas pela agregação.
assert df_resumo_itens_por_pedido['order_id'].is_unique
assert df_resumo_itens_por_pedido['order_id'].notna().all()

assert (
    len(df_resumo_itens_por_pedido)
    == df_itens_pedidos['order_id'].nunique()
)

assert df_resumo_itens_por_pedido[
    'quantidade_itens'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_produtos'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_vendedores'
].ge(1).all()


In [9]:
# Adiciona o resumo dos itens aos pedidos, preservando inclusive pedidos sem itens.
df_pedidos = (
    df_pedidos_clientes
    .merge(
        df_resumo_itens_por_pedido,
        on='order_id',
        how='left',
        validate='one_to_one'
    )
    .copy()
)

assert df_resumo_itens_por_pedido['order_id'].is_unique
assert df_resumo_itens_por_pedido['order_id'].notna().all()

assert (
    len(df_resumo_itens_por_pedido)
    == df_itens_pedidos['order_id'].nunique()
)

assert df_resumo_itens_por_pedido[
    'quantidade_itens'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_produtos'
].ge(1).all()

assert df_resumo_itens_por_pedido[
    'quantidade_vendedores'
].ge(1).all()

df_pedidos.head()


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,quantidade_itens,quantidade_produtos,quantidade_vendedores,quantidade_categorias,valor_produtos,valor_frete,peso_total_g,volume_total_cm3,possui_dimensoes_ausentes,possui_peso_ausente,possui_categoria_ausente,valor_total,pct_frete,pedido_um_produto,pedido_um_vendedor
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,1.0,29.99,8.72,500.0,1976.0,False,False,False,38.71,0.290764,True,True
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,1.0,118.70,22.76,400.0,4693.0,False,False,False,141.46,0.191744,True,True
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,1.0,159.90,19.22,420.0,9576.0,False,False,False,179.12,0.120200,True,True
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,1.0,45.00,27.20,450.0,6000.0,False,False,False,72.20,0.604444,True,True
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,1.0,19.90,8.72,250.0,11475.0,False,False,False,28.62,0.438191,True,True


In [10]:
# Acrescenta as avaliações tratadas e forma a base analítica no nível do pedido.
df_base_analitica = (
    df_pedidos
    .merge(
        df_reviews_por_pedido,
        on='order_id',
        how='left',
        validate='one_to_one'
    )
    .copy()
)

df_base_analitica['possui_itens'] = (
    df_base_analitica['quantidade_itens'].notna()
)

df_base_analitica['possui_avaliacao'] = (
    df_base_analitica['review_score'].notna()
)

df_base_analitica['pedido_entregue'] = (
    df_base_analitica['order_status'].eq('delivered')
)

df_base_analitica['delivery_delay_days'] = (
    df_base_analitica['order_delivered_customer_date']
    - df_base_analitica['order_estimated_delivery_date']
).dt.days

df_base_analitica['is_late'] = pd.Series(
    pd.NA,
    index=df_base_analitica.index,
    dtype='boolean'
)

mascara_prazo_disponivel = (
    df_base_analitica['delivery_delay_days'].notna()
)

df_base_analitica.loc[
    mascara_prazo_disponivel,
    'is_late'
] = (
    df_base_analitica.loc[
        mascara_prazo_disponivel,
        'delivery_delay_days'
    ].gt(0)
)

print('Pedidos sem itens:',df_base_analitica['possui_itens'].eq(False).sum())

print('Pedidos sem avaliação:',df_base_analitica['possui_avaliacao'].eq(False).sum())

print('Pedidos sem cálculo de atraso:',df_base_analitica['is_late'].isna().sum())

df_base_analitica.head()


Pedidos sem itens: 775
Pedidos sem avaliação: 591
Pedidos sem cálculo de atraso: 2965


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,quantidade_itens,quantidade_produtos,quantidade_vendedores,quantidade_categorias,valor_produtos,valor_frete,peso_total_g,volume_total_cm3,possui_dimensoes_ausentes,possui_peso_ausente,possui_categoria_ausente,valor_total,pct_frete,pedido_um_produto,pedido_um_vendedor,review_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,review_event_timestamp,review_ruim,review_neutra,review_boa,possui_itens,possui_avaliacao,pedido_entregue,delivery_delay_days,is_late
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,1.0,1.0,1.0,1.0,29.99,8.72,500.0,1976.0,False,False,False,38.71,0.290764,True,True,a54f0611adc9ed256b57ede6b6eb5114,4.0,NaN,"Não testei o produto ainda, mas ele veio corre...",2017-10-11,2017-10-12 03:43:48,2017-10-12 03:43:48,False,False,True,True,True,True,-8.0,False
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,1.0,1.0,1.0,1.0,118.70,22.76,400.0,4693.0,False,False,False,141.46,0.191744,True,True,8d5266042046a06655c8db133d120ba5,4.0,Muito boa a loja,Muito bom o produto.,2018-08-08,2018-08-08 18:37:50,2018-08-08 18:37:50,False,False,True,True,True,True,-6.0,False
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,1.0,1.0,1.0,1.0,159.90,19.22,420.0,9576.0,False,False,False,179.12,0.120200,True,True,e73b67b67587f7644d5bd1a52deb1b01,5.0,NaN,NaN,2018-08-18,2018-08-22 19:07:58,2018-08-22 19:07:58,False,False,True,True,True,True,-18.0,False
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,1.0,1.0,1.0,1.0,45.00,27.20,450.0,6000.0,False,False,False,72.20,0.604444,True,True,359d03e676b3c069f62cadba8dd3f6e8,5.0,NaN,O produto foi exatamente o que eu esperava e e...,2017-12-03,2017-12-05 19:21:58,2017-12-05 19:21:58,False,False,True,True,True,True,-13.0,False
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,1.0,1.0,1.0,1.0,19.90,8.72,250.0,11475.0,False,False,False,28.62,0.438191,True,True,e50934924e227544ba8246aeb3770dd4,5.0,NaN,NaN,2018-02-17,2018-02-18 13:02:51,2018-02-18 13:02:51,False,False,True,True,True,True,-10.0,False


In [11]:
display(
    pd.DataFrame({
        'coluna': df_base_analitica.columns,
        'tipo': df_base_analitica.dtypes.astype(str).values,
        'quantidade_nao_nulos': df_base_analitica.notna().sum().values,
        'quantidade_nulos': df_base_analitica.isna().sum().values
    })
)


,coluna,tipo,quantidade_nao_nulos,quantidade_nulos
0,order_id,str,99441,0
1,customer_id,str,99441,0
2,order_status,str,99441,0
3,order_purchase_timestamp,datetime64[ns],99441,0
4,order_approved_at,datetime64[ns],99281,160
5,order_delivered_carrier_date,datetime64[ns],97658,1783
6,order_delivered_customer_date,datetime64[ns],96476,2965
7,order_estimated_delivery_date,datetime64[ns],99441,0
8,customer_unique_id,str,99441,0
9,customer_zip_code_prefix,int64,99441,0


In [12]:
display(
    pd.DataFrame({
        'coluna': df_itens_pedidos.columns,
        'tipo': df_itens_pedidos.dtypes.astype(str).values,
        'quantidade_nao_nulos': df_itens_pedidos.notna().sum().values,
        'quantidade_nulos': df_itens_pedidos.isna().sum().values
    })
)


,coluna,tipo,quantidade_nao_nulos,quantidade_nulos
0,order_id,str,112650,0
1,order_item_id,int64,112650,0
2,product_id,str,112650,0
3,seller_id,str,112650,0
4,shipping_limit_date,datetime64[ns],112650,0
5,price,float64,112650,0
6,freight_value,float64,112650,0
7,product_category_name,str,111047,1603
8,product_name_lenght,float64,111047,1603
9,product_description_lenght,float64,111047,1603


## Exportações

Salva e confere as bases detalhadas em data/processed.

In [13]:
CAMINHO_PROCESSADOS = RAIZ_PROJETO / 'data' / 'processed'
CAMINHO_PROCESSADOS.mkdir(parents=True, exist_ok=True)

df_base_analitica.to_parquet(
    f'{CAMINHO_PROCESSADOS}/base_analitica_pedidos.parquet',
    index=False
)

df_itens_pedidos.to_parquet(
    f'{CAMINHO_PROCESSADOS}/itens_pedidos.parquet',
    index=False
)
pd.testing.assert_frame_equal(
    df_base_analitica, pd.read_parquet(CAMINHO_PROCESSADOS / 'base_analitica_pedidos.parquet'),
    check_exact=True
)
pd.testing.assert_frame_equal(
    df_itens_pedidos, pd.read_parquet(CAMINHO_PROCESSADOS / 'itens_pedidos.parquet'),
    check_exact=True
)
